# Cellpose Segmentation with Enhanced QC and Export Options

**Modified version of Cellpose-SAM notebook with additional features for tracking pipelines**

Original notebook by Marius Pachitariu, Michael Rariden, Carsen Stringer  
[Original paper](https://www.biorxiv.org/content/10.1101/2025.04.28.651001v1) | [Cellpose repository](https://github.com/MouseLand/cellpose)

Adapted from notebook by Pradeep Rajasekhar, inspired by [ZeroCostDL4Mic](https://github.com/HenriquesLab/ZeroCostDL4Mic/wiki).

---

## 🆕 My Modifications & Improvements

### 1. Stack Export for TrackMate-FIJI
- Added creation of 2D TIFF stack (`masks_stack.tif`) with proper ImageJ metadata
- Individual masks still saved as before, plus stack format
- Proper TYX axis ordering for time-series analysis

### 2. ultrack Preparation
Enhanced code prepares complete ultrack workflow inputs:

**Essential ultrack files generated:**
- Segmentation masks (2D stack)
- Detection properties CSV (centroids, areas, bounding boxes)
- Raw image stack for appearance-based tracking
- Optical flow fields from Cellpose flows for motion prediction
- Detection confidence scores from Cellpose probabilities

### 3. Additional Improvements
- Better file sorting to ensure temporal order
- Enhanced visualization with side-by-side comparisons
- More comprehensive analysis plots
- Detailed summary information
- Error handling for different image formats

### 4. Quality Control Features
- [List your QC additions here]
- Interactive parameter adjustment
- Visual validation of segmentation results

---

## Output Structure

After running, you'll get an `ultrack_data/` folder with:
- `masks_stack.tif` - Segmentation results
- `detections.csv` - Object properties and centroids
- `flow_stack.tif` - Motion vectors for tracking
- `ultrack_config_info.txt` - Summary of generated data


---

## Usage

### Google Colab (Recommended)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/cellpose-coordinate-extraction/blob/main/notebooks/cellpose_enhanced_qc.ipynb)

*Optimized for Google Colab. Can also run locally with environment.yml*

### Local
```bash
conda env create -f environment.yml
jupyter notebook notebooks/01_cellpose_ultrack_qc_colab.ipynb
```
*Note: Ignore Colab-specific cells (Google Drive mounting, pip installs)*

## Requirements

See `environment.yml` for full dependencies. Key packages:
- cellpose-sam
- scikit-image
- tifffile (for ImageJ-compatible TIFF export)
- ultrack (optional, for tracking)

## Citation

If you use this modified workflow, please cite both:

**This modified version:**
```
Virginia Silio. (2025). Cellpose Enhanced QC and Export Pipeline.
GitHub. https://github.com/virsicas/cellpose2coordinate
```

**Original Cellpose-SAM:**
```
Pachitariu, M., Rariden, M., & Stringer, C. (2025). 
Cellpose-SAM: superhuman generalization for cellular segmentation. 
bioRxiv. https://doi.org/10.1101/2025.04.28.651001
```

## License

This modified version: CC BY 4.0

Original Cellpose code: BSD-3-Clause (see [Cellpose repository](https://github.com/MouseLand/cellpose))

## Acknowledgments

Built upon Cellpose-SAM by Pachitariu et al., adapted from Pradeep Rajasekhar's notebook, inspired by ZeroCostDL4Mic.

Modifications focus on improving export compatibility with tracking tools and enhancing quality control workflows.

### Make sure you have GPU access enabled by going to Runtime -> Change Runtime Type -> Hardware accelerator and selecting GPU


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Install Cellpose-SAM


In [ ]:
%pip install git+https://www.github.com/mouseland/cellpose.git

Check GPU and instantiate model - will download weights.

In [ ]:
import numpy as np
from cellpose import models, core, io
from pathlib import Path
import matplotlib.pyplot as plt
import glob
from tifffile import imwrite
import os
import pandas as pd


io.logger_setup() # run this to get printing of progress

#Check if colab notebook instance has GPU access
if core.use_gpu()==False:
  raise ImportError("No GPU access, change your runtime")

Input directory with your model and path to images :

In [ ]:
# 1. path to your trained model (.pth file or model folder)
model_path = "path"                      # <-- update this

# 2. load your pretrained model
model = models.CellposeModel(
    gpu=True,                # set False if no GPU
    pretrained_model=model_path
)

# 3. folder with test images
test_dir = "path"                       # <-- update this
test_files = sorted(glob.glob(f"{test_dir}/*.tif"))  # Sort to ensure proper time order
imgs = [io.imread(f) for f in test_files]

print(f"Found {len(test_files)} images to process")

Run model as batch, check diameter

In [ ]:
# 4. run inference with your model
masks, flows, styles = model.eval(imgs, diameter=75) # <-- update this to change diameter

# 5. Enhanced saving with both individual masks and stacks
save_dir = "path"                       # <-- Change this
os.makedirs(save_dir, exist_ok=True)

# Save individual masks (as before)
individual_masks = []
for f, mask in zip(test_files, masks):
    base, _ = os.path.splitext(os.path.basename(f))
    out_path = os.path.join(save_dir, base + "_mask.tif")
    imwrite(out_path, mask.astype("uint16"))
    individual_masks.append(mask.astype("uint16"))

Save mask_stack for TrackMate-FIJI

In [ ]:
# NEW: Save as a 3D stack for TrackMate-FIJI
stack_path = os.path.join(save_dir, "masks_stack.tif") 
mask_stack = np.stack(individual_masks, axis=0)
imwrite(stack_path, mask_stack, imagej=True, metadata={'axes': 'TYX'})
print(f"Saved mask stack to: {stack_path}")

Data for Ultrack

In [ ]:
# NEW: Prepare data for ultrack
def prepare_ultrack_data(imgs, masks, flows, test_files, save_dir):
    """
    Prepare necessary outputs for ultrack tracking:
    1. Raw images stack
    2. Masks stack
    3. Flow fields (for motion estimation)
    4. Detection confidence scores
    5. Centroids and properties CSV
    """

    # Create ultrack subdirectory
    ultrack_dir = os.path.join(save_dir, "ultrack_data")
    os.makedirs(ultrack_dir, exist_ok=True)

    # 1. Save raw images as stack
    if len(imgs) > 0:
        # Handle different image formats
        if imgs[0].ndim == 3 and imgs[0].shape[0] <= 3:  # (C, H, W)
            img_stack = np.stack(imgs, axis=0)  # (T, C, H, W)
        else:
            img_stack = np.stack(imgs, axis=0)  # (T, H, W)

        raw_stack_path = os.path.join(ultrack_dir, "raw_images_stack.tif")
        imwrite(raw_stack_path, img_stack.astype(np.float32), imagej=True)
        print(f"Saved raw images stack to: {raw_stack_path}")

    # 2. Save masks (already done above, but copy to ultrack folder)
    ultrack_mask_path = os.path.join(ultrack_dir, "masks_stack.tif")
    imwrite(ultrack_mask_path, mask_stack, imagej=True, metadata={'axes': 'TYX'})

    # 3. Save flow fields (motion vectors)
    if len(flows) > 0:
        flow_stack = np.stack([f[0] for f in flows], axis=0)  # flows[0] contains the motion vectors
        flow_path = os.path.join(ultrack_dir, "flow_stack.tif")
        imwrite(flow_path, flow_stack.astype(np.float32), imagej=True)
        print(f"Saved flow fields to: {flow_path}")

    # 4. Extract detection confidence (from cellpose probabilities)
    if len(flows) > 0:
        prob_stack = np.stack([f[2] for f in flows], axis=0)  # flows[2] contains probabilities
        prob_path = os.path.join(ultrack_dir, "probabilities_stack.tif")
        imwrite(prob_path, prob_stack.astype(np.float32), imagej=True)
        print(f"Saved detection probabilities to: {prob_path}")

    # 5. Extract centroids and properties for each timepoint
    detection_data = []

    for t, (mask, filename) in enumerate(zip(masks, test_files)):
        n_objects = mask.max()

        for obj_id in range(1, n_objects + 1):
            obj_mask = (mask == obj_id)
            if obj_mask.sum() == 0:
                continue

            # Calculate properties
            coords = np.where(obj_mask)
            centroid_y = np.mean(coords[0])
            centroid_x = np.mean(coords[1])
            area = obj_mask.sum()

            # Bounding box
            min_y, max_y = coords[0].min(), coords[0].max()
            min_x, max_x = coords[1].min(), coords[1].max()

            detection_data.append({
                'timepoint': t,
                'object_id': obj_id,
                'centroid_x': centroid_x,
                'centroid_y': centroid_y,
                'area': area,
                'bbox_min_x': min_x,
                'bbox_max_x': max_x,
                'bbox_min_y': min_y,
                'bbox_max_y': max_y,
                'filename': os.path.basename(filename)
            })

    # Save as CSV
    df = pd.DataFrame(detection_data)
    csv_path = os.path.join(ultrack_dir, "detections.csv")
    df.to_csv(csv_path, index=False)
    print(f"Saved detection properties to: {csv_path}")

    # Create a summary file for ultrack configuration
    summary_path = os.path.join(ultrack_dir, "ultrack_config_info.txt")
    with open(summary_path, 'w') as f:
        f.write("ultrack Configuration Information\n")
        f.write("=" * 40 + "\n\n")
        f.write(f"Number of timepoints: {len(masks)}\n")
        f.write(f"Image dimensions: {masks[0].shape}\n")
        f.write(f"Total detections: {len(detection_data)}\n")
        f.write(f"Average objects per frame: {len(detection_data)/len(masks):.1f}\n\n")
        f.write("Files generated:\n")
        f.write("- raw_images_stack.tif: Original images\n")
        f.write("- masks_stack.tif: Segmentation masks\n")
        f.write("- flow_stack.tif: Optical flow vectors\n")
        f.write("- probabilities_stack.tif: Detection confidence\n")
        f.write("- detections.csv: Object properties and centroids\n\n")
        f.write("For ultrack, you'll typically need:\n")
        f.write("1. The segmentation masks (masks_stack.tif)\n")
        f.write("2. The detection properties (detections.csv)\n")
        f.write("3. Optionally, the flow fields for motion prediction\n")

    print(f"Created ultrack summary at: {summary_path}")
    return ultrack_dir


In [ ]:
# Prepare ultrack data
ultrack_dir = prepare_ultrack_data(imgs, masks, flows, test_files, save_dir)


Visualization

In [ ]:
# Visualization function 
def show_image_with_mask(img, mask, alpha=0.3, title=""):
    plt.figure(figsize=(12, 6))

    # Original image
    plt.subplot(1, 2, 1)
    if img.ndim == 3:  # (C, H, W)
        if img.shape[0] == 1:  # single channel
            plt.imshow(img[0], cmap='gray')
        elif img.shape[0] == 2:  # two channels
            img_to_show = np.zeros((img.shape[1], img.shape[2], 3), dtype=img.dtype)
            img_to_show[..., 0] = img[0]  # red
            img_to_show[..., 1] = img[1]  # green
            plt.imshow(img_to_show)
        else:
            # Take first 3 channels if more
            plt.imshow(np.transpose(img[:3], (1, 2, 0)))
    else:
        plt.imshow(img, cmap='gray')
    plt.title(f"Original {title}")
    plt.axis('off')

    # Overlay
    plt.subplot(1, 2, 2)
    if img.ndim == 3:
        if img.shape[0] == 1:
            plt.imshow(img[0], cmap='gray')
        elif img.shape[0] == 2:
            img_to_show = np.zeros((img.shape[1], img.shape[2], 3), dtype=img.dtype)
            img_to_show[..., 0] = img[0]
            img_to_show[..., 1] = img[1]
            plt.imshow(img_to_show)
        else:
            plt.imshow(np.transpose(img[:3], (1, 2, 0)))
    else:
        plt.imshow(img, cmap='gray')

    # Create colored mask overlay
    colored_mask = np.zeros((*mask.shape, 4))
    for i in range(1, mask.max() + 1):
        color = plt.cm.tab20(i % 20)  # Cycle through colors
        colored_mask[mask == i] = color
    colored_mask[..., 3] = alpha * (mask > 0)  # Set alpha

    plt.imshow(colored_mask)
    plt.title(f"Segmentation {title}")
    plt.axis('off')
    plt.tight_layout()
    plt.show()




In [ ]:

file_labels = []
n_objects_list = []
mean_areas = []
total_areas = []

print("\nPer-image analysis:")
print("-" * 50)

for f, mask in zip(test_files, masks):
    n_objects = mask.max()
    areas = [(mask == i).sum() for i in range(1, n_objects+1)]
    mean_area = np.mean(areas) if areas else 0
    total_area = sum(areas) if areas else 0

    # Extract the time label from filename
    base = os.path.basename(f)
    t_label = base.split('_')[-1].split('.')[0]
    file_labels.append(t_label)

    n_objects_list.append(n_objects)
    mean_areas.append(mean_area)
    total_areas.append(total_area)

    print(f"{base}: {n_objects:3d} objects, mean area = {mean_area:6.1f} px, total area = {total_area:8.0f} px")


Check plot masks

In [ ]:

file_labels = []
n_objects_list = []
mean_areas = []

for f, mask in zip(test_files, masks):
    n_objects = mask.max()
    areas = [(mask == i).sum() for i in range(1, n_objects+1)]
    mean_area = np.mean(areas) if areas else 0

    # Extract the "T..." part from filename
    base = os.path.basename(f)
    t_label = base.split('_')[-1].split('.')[0]  # takes last underscore part before extension
    file_labels.append(t_label)

    n_objects_list.append(n_objects)
    mean_areas.append(mean_area)

    print(f"{base}: {n_objects} objects, mean area = {mean_area:.1f} px")


In [ ]:
# Enhanced plotting
plt.figure(figsize=(15, 10))

# Plot 1: Number of objects
plt.subplot(2, 2, 1)
plt.bar(file_labels, n_objects_list, alpha=0.7)
plt.xlabel("Time point")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.ylabel("Number of objects")
plt.title("Objects detected per timepoint")
plt.grid(True, alpha=0.3)

# Plot 2: Mean area
plt.subplot(2, 2, 2)
plt.bar(file_labels, mean_areas, alpha=0.7, color='orange')
plt.xlabel("Time point")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.ylabel("Mean object area (px)")
plt.title("Mean object area per timepoint")
plt.grid(True, alpha=0.3)

# Plot 3: Total area
plt.subplot(2, 2, 3)
plt.bar(file_labels, total_areas, alpha=0.7, color='green')
plt.xlabel("Time point")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.ylabel("Total area (px)")
plt.title("Total segmented area per timepoint")
plt.grid(True, alpha=0.3)

# Plot 4: Time series line plot
plt.subplot(2, 2, 4)
x_numeric = range(len(file_labels))
plt.plot(x_numeric, n_objects_list, 'o-', label='Object count', alpha=0.7)
plt.plot(x_numeric, np.array(mean_areas)/10, 's-', label='Mean area (/10)', alpha=0.7)
plt.xlabel("Time point")
plt.ylabel("Value")
plt.title("Time series trends")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Display sample results (first 3 images)
print(f"\nDisplaying sample results...")
for i, (img, mask, filename) in enumerate(zip(imgs[:3], masks[:3], test_files[:3])):
    show_image_with_mask(img, mask, title=f"({i+1}) {os.path.basename(filename)}")

print(f"\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Processed {len(imgs)} images")
print(f"Individual masks saved to: {save_dir}")
print(f"Mask stack for TrackMate saved to: {stack_path}")
print(f"ultrack-ready data saved to: {ultrack_dir}")
print(f"Total objects across all timepoints: {sum(n_objects_list)}")
print(f"Average objects per timepoint: {np.mean(n_objects_list):.1f}")
print("="*60)